<a href="https://colab.research.google.com/github/xkzy/pdf_scan_merge/blob/main/PDF_Merge_All_Pages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF Crop & Grid Merge — All Pages → One Page

Upload a PDF where **each page is one scanned document page**. Every page is rendered, its
document boundary is **detected and cropped** (perspective-corrected via a 4-point contour),
its background is **flattened toward white**, and all cropped pages are then **packed into a
grid on a single A4 canvas** — no overlaying, no blending, each page keeps its own space.

- **Crop**: edge detection finds the document's 4 corners on each page and deskews/crops to
  just that region. Pages where no clean 4-point boundary is found fall back to the full
  rendered page, so nothing is ever dropped.
- **Background removal**: after cropping, brightness-only correction (HSV 'V' channel) flattens
  any remaining shadow/tint toward white — hue and saturation are untouched, so document colors
  don't shift.
- **Grid placement**: pages are arranged in an auto-computed grid that best fills the A4 canvas
  for the given page count. Each cropped page is placed at its **true, unscaled pixel size**,
  centered in its own cell — never stretched or shrunk. If a page is larger than its cell, it
  is still placed at true size and a warning is printed (it may overlap neighboring pages).
- Default rendering resolution: **150–600 DPI**, calculated from the first page (see below).


In [ ]:
!pip -q install pymupdf pillow numpy opencv-python


In [ ]:
from google.colab import files

uploaded = files.upload()
input_pdf = next(iter(uploaded))
print(f"Input: {input_pdf}")


In [ ]:
import fitz  # PyMuPDF
from PIL import Image, ImageFilter
import numpy as np
import cv2
import io
import math

# ---------------- Configuration ----------------
REMOVE_BACKGROUND = True          # Flatten scan tint/shadow toward white after cropping
BG_BLUR_RADIUS = 25               # Blur radius used to estimate background for removal
ENABLE_DOCUMENT_CROP = True       # Detect + crop/deskew each page to its document boundary
MIN_CONTOUR_AREA_RATIO = 0.02     # Ignore candidate contours smaller than this fraction of the page
                                   # (kept low so small objects like ID cards, which cover only a
                                   # small fraction of an A4 scan, aren't rejected)
GRID_PADDING_RATIO = 0.015        # Padding between grid cells, as a fraction of the canvas' shorter side
A4_ORIENTATION = "portrait"       # "portrait", "landscape", or "auto" (auto matches the first page)
output_pdf = "cropped_grid_merged.pdf"
A4_WIDTH_MM = 210
A4_HEIGHT_MM = 297
TARGET_SHORTER_DIMENSION_PIXELS = 2400

# ---------------- Dynamic DPI from first page ----------------
_doc_temp = fitz.open(input_pdf)
if len(_doc_temp) == 0:
    _doc_temp.close()
    raise ValueError("The PDF contains no pages.")

_first_page_width_pts = _doc_temp[0].rect.width
_first_page_height_pts = _doc_temp[0].rect.height
_doc_temp.close()

_shortest_dim_pts = min(_first_page_width_pts, _first_page_height_pts)
if _shortest_dim_pts == 0:
    DPI = 150
else:
    DPI = round((TARGET_SHORTER_DIMENSION_PIXELS * 72) / _shortest_dim_pts)
    DPI = max(150, min(DPI, 600))
print(f"Rendering DPI: {DPI}")

# ---------------- Rendering ----------------
def render_page(page, dpi):
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

def remove_scan_background(img, blur_radius=25):
    """Flatten uneven scan background (shadow/paper texture) toward white
    WITHOUT changing document color. Only brightness (HSV 'V') is corrected:
    a blurred version of V is used as a local background estimate and divided
    out. Hue and saturation are left untouched, preserving ink/stamp/highlight
    colors exactly as scanned.
    """
    hsv = img.convert("HSV")
    h, s, v = hsv.split()

    v_arr = np.asarray(v).astype(np.float32)
    v_bg = np.asarray(v.filter(ImageFilter.GaussianBlur(blur_radius))).astype(np.float32)
    v_bg = np.clip(v_bg, 1, 255)

    v_norm = (v_arr / v_bg) * 255.0
    v_norm = np.clip(v_norm, 0, 255).astype(np.uint8)

    return Image.merge("HSV", (h, s, Image.fromarray(v_norm))).convert("RGB")

# ---------------- Document boundary detection & crop ----------------
def order_points(pts):
    # top-left, top-right, bottom-right, bottom-left
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth = max(int(widthA), int(widthB))

    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightA), int(heightB))

    if maxWidth < 1 or maxHeight < 1:
        return None

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    return cv2.warpPerspective(image, M, (maxWidth, maxHeight))

def crop_to_document(img_pil, page_number):
    """Detect the scanned object's boundary (document page or small item like
    an ID card) and crop/deskew to it. Falls back to the full, uncropped page
    if no clean boundary is found, so a page is never dropped from the merge."""
    image_cv = np.array(img_pil)
    gray = cv2.cvtColor(image_cv, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(gray, 50, 150)
    edged = cv2.dilate(edged, None, iterations=2)
    edged = cv2.morphologyEx(edged, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

    # RETR_EXTERNAL only: avoids latching onto internal contours (e.g. the
    # photo or text printed on a card) instead of the object's outer edge.
    cnts = cv2.findContours(edged.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    cnts = sorted(cnts, key=cv2.contourArea, reverse=True)[:5]

    img_area = image_cv.shape[0] * image_cv.shape[1]
    corner_pts = None
    for c in cnts:
        if cv2.contourArea(c) < MIN_CONTOUR_AREA_RATIO * img_area:
            continue
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4:
            corner_pts = approx.reshape(4, 2).astype("float32")
        else:
            # Rounded corners (e.g. ID cards) or a noisy outline rarely
            # simplify to exactly 4 points - use the rotated bounding box
            # of the contour instead, which still captures a tight crop.
            corner_pts = cv2.boxPoints(cv2.minAreaRect(c)).astype("float32")
        break

    if corner_pts is None:
        print(f"  Page {page_number}: no clean object boundary found - using full page.")
        return img_pil

    warped = four_point_transform(image_cv, corner_pts)
    if warped is None:
        print(f"  Page {page_number}: degenerate crop - using full page.")
        return img_pil
    return Image.fromarray(warped)

def prepare_page(page, page_number, dpi):
    img = render_page(page, dpi)
    if ENABLE_DOCUMENT_CROP:
        img = crop_to_document(img, page_number)
    if REMOVE_BACKGROUND:
        img = remove_scan_background(img, BG_BLUR_RADIUS)
    return img

# ---------------- A4 canvas & grid packing ----------------
def a4_pixel_size(dpi, orientation, first_page_size):
    w = round(dpi * A4_WIDTH_MM / 25.4)
    h = round(dpi * A4_HEIGHT_MM / 25.4)
    if orientation == "landscape":
        w, h = max(w, h), min(w, h)
    elif orientation == "portrait":
        w, h = min(w, h), max(w, h)
    else:  # auto: match the first page's orientation
        fw, fh = first_page_size
        if fw > fh:
            w, h = max(w, h), min(w, h)
        else:
            w, h = min(w, h), max(w, h)
    return (w, h)

def compute_grid(n, canvas_w, canvas_h):
    """Choose rows x cols that maximize the smallest resulting cell dimension,
    so the grid fills the A4 canvas as densely as possible for n pages."""
    best = None
    for cols in range(1, n + 1):
        rows = math.ceil(n / cols)
        cell_w = canvas_w / cols
        cell_h = canvas_h / rows
        score = min(cell_w, cell_h)
        if best is None or score > best[0]:
            best = (score, cols, rows)
    return best[1], best[2]

def paste_true_size_centered(canvas, img, cell_box, page_number):
    """Paste img onto canvas at its native (unscaled) size, centered within
    cell_box = (x0, y0, inner_w, inner_h). If img is larger than the cell,
    it is pasted as-is (may overlap neighboring cells) and a warning is
    printed instead of silently scaling it down."""
    x0, y0, inner_w, inner_h = cell_box
    iw, ih = img.size

    if iw > inner_w or ih > inner_h:
        print(f"  Warning: page {page_number} ({iw}x{ih}px) is larger than its "
              f"grid cell ({inner_w}x{inner_h}px) and may overlap neighboring pages.")

    offset_x = x0 + (inner_w - iw) // 2
    offset_y = y0 + (inner_h - ih) // 2
    canvas.paste(img, (offset_x, offset_y))

def build_grid_canvas(pages, canvas_size):
    n = len(pages)
    cw, ch = canvas_size
    cols, rows = compute_grid(n, cw, ch)
    print(f"Grid layout: {cols} cols x {rows} rows for {n} page(s)")

    pad = max(1, round(min(cw, ch) * GRID_PADDING_RATIO))
    cell_w = cw // cols
    cell_h = ch // rows

    canvas = Image.new("RGB", canvas_size, (255, 255, 255))
    for idx, page_img in enumerate(pages):
        r, c = divmod(idx, cols)
        cell_x0, cell_y0 = c * cell_w, r * cell_h
        inner_w = max(1, cell_w - 2 * pad)
        inner_h = max(1, cell_h - 2 * pad)
        paste_true_size_centered(canvas, page_img, (cell_x0 + pad, cell_y0 + pad, inner_w, inner_h), idx + 1)
    return canvas

# ---------------- Main pipeline ----------------
doc = fitz.open(input_pdf)
print(f"Pages: {len(doc)}")

prepared_pages = []
for i in range(len(doc)):
    print(f"Processing page {i + 1}/{len(doc)}...")
    prepared_pages.append(prepare_page(doc[i], i + 1, DPI))

base_size = a4_pixel_size(DPI, A4_ORIENTATION, prepared_pages[0].size)
print(f"Output A4 canvas: {base_size[0]} x {base_size[1]} px at {DPI} DPI")

result = build_grid_canvas(prepared_pages, base_size)
doc.close()

# Save as a single-page A4 PDF.
result.save(output_pdf, "PDF", resolution=DPI)

print(f"\nCreated: {output_pdf}")
print(f"Output size: {result.width} x {result.height} pixels")



In [ ]:
from IPython.display import display
display(result)


In [ ]:
files.download(output_pdf)
